# PA2 Notebook 3: Evaluation

Created from kaggle.com/benchmarks/tasks/new so that `kaggle_benchmarks` is available (no GPU needed,
internet on). Add the output of notebook 2 as input.

The judge (gemini-3.6-flash) compares DPO vs SFT and LoRA vs base using the saved pair files. It sends 5 pairs
per request with a 5 second wait, retries up to 3 times, and saves results after every batch so it can resume.

In [15]:
# clone the repo (needs internet on)
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

/kaggle/working/cs5326-PA2-27100015
1d054c5 (HEAD -> main, origin/main, origin/HEAD) Add report draft (training sections complete; results pending)


In [16]:
import subprocess
def sh(cmd):
    """run a shell command, print its output, and stop if it fails"""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode} (see output above): {cmd}")

In [17]:
import kaggle_benchmarks as kbench
JUDGE = "google/gemini-3.6-flash"  # judge model
print(sorted(kbench.llms))

['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-haiku-5-5', 'anthropic/claude-opus-4-5@20251101', 'anthropic/claude-opus-4-6@default', 'anthropic/claude-opus-4-7@default', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-opus-5-5@default', 'anthropic/claude-opus-5@default', 'anthropic/claude-sonnet-4-6@default', 'anthropic/claude-sonnet-5-5@default', 'anthropic/claude-sonnet-5@default', 'deepseek-ai/deepseek-r1-0528', 'google/gemini-2.5-flash', 'google/gemini-2.5-pro', 'google/gemini-3-flash-preview', 'google/gemini-3.1-flash-lite-preview', 'google/gemini-3.1-pro-preview', 'google/gemini-3.5-flash-lite', 'google/gemini-3.6-flash', 'google/gemini-3.8-flash', 'google/gemini-4-argon', 'google/gemma-4-26b-a4b', 'google/gemma-4-31b', 'openai/gpt-5.4-2026-03-05', 'openai/gpt-5.4-mini-2026-03-17', 'openai/gpt-5.4-nano-2026-03-17', 'openai/gpt-5.5-2026-04-23', 'openai/gpt-5.6-luna', 'openai/gpt-5.6-sol', 'openai/gpt-5.6-terra', 'openai/gpt-6-astra', 'openai/gpt-6-luna', 'openai

In [18]:
# helper to find files from the attached notebook output
import glob
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the saved notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [19]:
import os, shutil
os.makedirs("outputs/eval", exist_ok=True)
for name in ("dpo_uf_vs_sft_pairs.jsonl", "dpo_hh_vs_sft_pairs.jsonl", "lora_vs_base_pairs.jsonl"):
    shutil.copy(f"outputs/generations_rp1.0/{name}", f"outputs/eval/{name}")
!wc -l outputs/eval/*_pairs.jsonl


    300 outputs/eval/dpo_hh_vs_sft_pairs.jsonl
    300 outputs/eval/dpo_uf_vs_sft_pairs.jsonl
     40 outputs/eval/lora_vs_base_pairs.jsonl
    640 total


In [20]:
# test the judge on 2 pairs first
sh("python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl "
   f"--output_path outputs/eval/smoke_judged.jsonl --judge_model {JUDGE} --limit 2 "
   "--batch_size 5 --sleep_seconds 5 --max_retries 3")
!head -c 2000 outputs/eval/smoke_judged_raw.jsonl
!rm -f outputs/eval/smoke_judged*

$ python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl --output_path outputs/eval/smoke_judged.jsonl --judge_model google/gemini-3.6-flash --limit 2 --batch_size 5 --sleep_seconds 5 --max_retries 3
[judge] mode=general judge=google/gemini-3.6-flash backend=kaggle
[judge] 2 pairs, 0 reused, 2 to judge
🧵pairwise_judge_0:
  👤 [User]: You are an impartial expert judge comparing two AI assistant responses to the same user instruction.

Judge which response is better overall. Consider helpfulness, correctness, instruction-following, concision when appropriate, clear formatting, and the absence of hallucinations or unsafe content.

Do not let the position of a response (A or B) or its length alone influence your decision. Use "tie" only when the responses are genuinely equal in quality.

There are 2 independent items below. Judge each one separately.

### Item 1
[Instruction]
Create a short, concise summary of the paper based on its abstract.

Few-shot l

## Judging

In [21]:
for name in ("dpo_uf_vs_sft", "dpo_hh_vs_sft", "lora_vs_base"):
    sh(f"python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/{name}_pairs.jsonl "
       f"--output_path outputs/eval/{name}_judged.jsonl --judge_model {JUDGE} "
       "--batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments")

$ python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_pairs.jsonl --output_path outputs/eval/dpo_uf_vs_sft_judged.jsonl --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments
[judge] mode=general judge=google/gemini-3.6-flash backend=kaggle
[judge] 300 pairs, 260 reused, 40 to judge
🧵pairwise_judge_0:
  👤 [User]: You are an impartial expert judge comparing two AI assistant responses to the same user instruction.

Judge which response is better overall. Consider helpfulness, correctness, instruction-following, concision when appropriate, clear formatting, and the absence of hallucinations or unsafe content.

Do not let the position of a response (A or B) or its length alone influence your decision. Use "tie" only when the responses are genuinely equal in quality.

There are 5 independent items below. Judge each one separately.

### Item 1
[Instruction]
Can you provide a list of health food brands owned by Hisp

In [24]:
import shutil
shutil.make_archive("/kaggle/working/evaluation_outputs", "zip", "outputs", "eval")
!ls -lh /kaggle/working/evaluation_outputs.zip


-rw-r--r-- 1 root root 344K Oct 18 14:22 /kaggle/working/evaluation_outputs.zip


In [25]:
sh("python scripts/collect_results.py --eval_dir outputs/eval --output outputs/eval/results.md")
!cd outputs && zip -q -r /kaggle/working/evaluation_outputs.zip eval
!ls -lh /kaggle/working/evaluation_outputs.zip

$ python scripts/collect_results.py --eval_dir outputs/eval --output outputs/eval/results.md
## DPO vs SFT (candidate perspective)

| Comparison | Criterion | Judge | N | Wins | Ties | Losses | Unknown | Failures | Win rate (ties=0.5) | Win rate (no ties) | Retries |
|---|---|---|---|---|---|---|---|---|---|---|---|
| dpo_hh vs sft (dpo_hh_vs_sft) | winner | google/gemini-3.6-flash | 300 | 139 | 57 | 104 | 0 | 0 | 55.8% | 57.2% | 0 |
| dpo_uf vs sft (dpo_uf_vs_sft) | winner | google/gemini-3.6-flash | 300 | 122 | 43 | 135 | 0 | 0 | 47.8% | 47.5% | 0 |
| lora_dpo vs smollm2_instruct (lora_vs_base) | style | google/gemini-3.6-flash | 40 | 21 | 18 | 1 | 0 | 0 | 75.0% | 95.5% | 0 |
| lora_dpo vs smollm2_instruct (lora_vs_base) | quality | google/gemini-3.6-flash | 40 | 12 | 2 | 26 | 0 | 0 | 32.5% | 31.6% | 0 |

## By category

| Comparison | Criterion | Category | Wins | Ties | Losses | Win rate (ties=0.5) |
|---|---|---|---|---|---|---|
| dpo_hh vs sft (dpo_hh_vs_sft) | winner | benign_he